In [1]:
import os
import logging, timeit
from btEngine2.DataLoader import DataLoader
from btEngine2.MarketData import MarketData
from btEngine2.TradingRule import TradingRule



# Define paths and configurations for DataLoader
ticker_csv_path = r'G:\Projects\BackTesting1.0\Data\Inputs\TickerList-Futs.csv'
save_directory = r"G:\Projects\BackTesting1.0\Data\Bloomberg\Futures"
helper_directory = r'G:\Projects\BackTesting1.0\Data\Bloomberg\HelperFiles'



# Define paths to auxiliary data for MarketData
tick_values_path = os.path.join(helper_directory, 'fut_val_pt.parquet')
fx_rates_path = os.path.join(helper_directory, 'fxHist.parquet')

# Initialize the MarketData
market_data = MarketData(
    base_directory=save_directory,
    tick_values_path=tick_values_path,
    fx_rates_path=fx_rates_path,
    instrument_type="Futures",
    n_threads=8,  # Number of threads for parallel data loading
    log_level=logging.ERROR  # Set to DEBUG for more detailed logs
)


In [2]:
tick = 'XTD1 Curncy'
# Access data for a specific ticker
try:
    test_df = market_data.get_ticker_data(tick)
    print(test_df)
except ValueError as e:
    print(e)

# Access all preprocessed data
all_data = market_data.get_data()
print(f"Total tickers loaded: {len(all_data)}")

# Access FX rates
fx_rates = market_data.get_fx_rates()
# Access tick values
tick_values = market_data.get_tick_values()
# Access asset classes
asset_classes = market_data.get_asset_classes()

#market_data = market_data.date_filter(start_date='01012010')

shape: (2_371, 14)
┌────────────┬────────┬────────┬────────┬───┬─────────┬─────────┬─────────────────┬────────────────┐
│ Date       ┆ Open   ┆ High   ┆ Low    ┆ … ┆ BadOHLC ┆ FX_Rate ┆ Tick_Value_Base ┆ Tick_Value_USD │
│ ---        ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---     ┆ ---     ┆ ---             ┆ ---            │
│ date       ┆ f64    ┆ f64    ┆ f64    ┆   ┆ bool    ┆ f64     ┆ f64             ┆ f64            │
╞════════════╪════════╪════════╪════════╪═══╪═════════╪═════════╪═════════════════╪════════════════╡
│ 2015-09-01 ┆ null   ┆ null   ┆ null   ┆ … ┆ false   ┆ 1.0     ┆ 100000.0        ┆ 100000.0       │
│ 2015-09-02 ┆ null   ┆ null   ┆ null   ┆ … ┆ false   ┆ 1.0     ┆ 100000.0        ┆ 100000.0       │
│ 2015-09-03 ┆ null   ┆ null   ┆ null   ┆ … ┆ false   ┆ 1.0     ┆ 100000.0        ┆ 100000.0       │
│ 2015-09-04 ┆ null   ┆ null   ┆ null   ┆ … ┆ false   ┆ 1.0     ┆ 100000.0        ┆ 100000.0       │
│ 2015-09-07 ┆ null   ┆ null   ┆ null   ┆ … ┆ false   ┆ 1.0     ┆ 100000

In [11]:

from btEngine2.Rules.Momentum.sbo_long import *

sboParams1 = {
    'X': 12,
    'N': 2,
    'r': 0.5,
    'atr_type': 'atr'
}

sboParamsPb = {
    'X': 12,
    'N': 2,
    'r': 0.5,
    'atr_type': 'atr',
    'lmt_days': 1,
    'lmt_atr_ratio': 0.5,
    'lmt_epsilon': 0.1
}


pSizeParams = {
    'AssetVol': 1_200_000,  # Target asset volatility in USD
    'VolLookBack': 125  # Lookback period for volatility calculation
}

pSizeParams2 = {
    'AssetVol': r'G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv',  # Target asset volatility in USD
    'VolLookBack': 125  # Lookback period for volatility calculation
}

pSizeParamsV1 = {
    'AssetVol': 10_000_000,  # Target asset volatility in USD
    'VolLookBack': 30  # Lookback period for volatility calculation
}

pSizeParams2V2 = {
    'AssetVol': r'G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv',  # Target asset volatility in USD
    'VolLookBack': 21  # Lookback period for volatility calculation
}

In [12]:
tRuleConst = TradingRule(
    market_data=market_data,
    trading_rule_function=sbo_long,
    trading_params=sboParams1,
    position_sizing_params=pSizeParams2V2,
    cont_rule=False,
    name_label='SBO1',
    assign_id=True
)

tRuleDynamic = TradingRule(
    market_data=market_data,
    trading_rule_function=sbo_long_pb,
    trading_params=sboParamsPb,
    position_sizing_params=pSizeParams2V2,
    cont_rule=False,
    name_label='SBO_PB'
)


2024-10-16 20:16:18,158 - TradingRule_sbo_long_17713ba5c2 - INFO - Initialized TradingRule for function 'sbo_long' with params hash '17713ba5c2'
2024-10-16 20:16:18,223 - TradingRule_sbo_long_17713ba5c2 - INFO - Loaded AssetVol from file: G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv
2024-10-16 20:16:18,225 - TradingRule_sbo_long_pb_2c33be5978 - INFO - Initialized TradingRule for function 'sbo_long_pb' with params hash '2c33be5978'
2024-10-16 20:16:18,226 - TradingRule_sbo_long_pb_2c33be5978 - INFO - Loaded AssetVol from file: G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv


In [ ]:
results = tRuleConst.backtest_all_assets(save=True)
results2 = tRuleDynamic.backtest_all_assets(save=True)

In [ ]:
filterac = ['stir', 'fi-eu', 'fi-us', 'comm-prec']
resdb_pb = tRuleDynamic.plot_equity(byac=True,totalsys=True, 
                                    filter_ac=filterac, start_date='01012015', end_date='01012017')

In [ ]:

resdb = tRuleConst.plot_equity(byac=True,totalsys=True, filter_ac=['stir', 'fi-eu', 'fi-us'])

In [ ]:

resdb_pb = tRuleDynamic.plot_equity(byac=True,totalsys=True)

In [ ]:
res = tRuleConst.plot_equity(byac=True, totalsys=True)

In [ ]:
stats_df = tRuleConst.calculate_statistics(byassets=True, totalsys=True, filter_assets=asset_classes['eq-divf'], start_date='01012024', by_actual_trade=True)
curr_plot = tRuleConst.plot_equity(byassets=True, totalsys=True, filter_assets=asset_classes['eq-divf'], start_date='01012024')
stats_df.to_clipboard()

In [ ]:
asset_classes

In [ ]:
stats_test = tRuleConst.calculate_statistics(byassets=True, by_actual_trade=True)

In [13]:
from btEngine2.Rules.MeanReversion.ratioMR import *

# Define the list of asset pairs you want to trade
pairs = [
    ('NQ1 Index', 'US1 Comdty')
]

# Set strategy parameters
strategy_params = {
    'pairs': pairs,
    'N': 5,
    'rsi_period': 3,
    'rsi_threshold': 10.0,
    'lmt_order': True,
    'lmt_day': 2,
    'lmt_day_only': False,
    'lmt_atr': 1,
    'lmt_epsilon': 0.1,
    'atr_period': 5,
    'atr_type': 'atr',
    'market_data': market_data  # Pass the MarketData instance
}

strategy_params2 = {
    'pairs': pairs,
    'N': 5,
    'rsi_period': 3,
    'rsi_threshold': 10.0,
    'market_data': market_data,  # Pass the MarketData instance
}

strategy_params3 = strategy_params2.copy()
strategy_params3['trend_filter'] = 120

strategy_params4 = strategy_params2.copy()
strategy_params4['oversold_cond'] = True

strategy_params5 = strategy_params3.copy()
strategy_params5['oversold_cond'] = True

# Create an instance of TradingRule
trading_rule_MR = TradingRule(
    market_data=market_data,
    trading_rule_function=ratioMR_rsi_long,
    trading_params=strategy_params,
    position_sizing_params=pSizeParamsV1,  # Define as needed
    incl_assets=['NQ1 Index', 'US1 Comdty'],  # Include all involved assets
    name_label='TestMR',
    strat_descr='RSI MR with next day lmt order close - 0.7 atr'
)

tRuleMRbasic = TradingRule(
    market_data=market_data,
    trading_rule_function=ratioMR_rsi_long,
    trading_params=strategy_params2,
    position_sizing_params=pSizeParamsV1,  # Define as needed
    incl_assets=['NQ1 Index', 'US1 Comdty'],  # Include all involved assets
    name_label='TestMRb1',
    strat_descr='RSI MR basic'
)

tRuleMRbasic_tf = TradingRule(
    market_data=market_data,
    trading_rule_function=ratioMR_rsi_long,
    trading_params=strategy_params3,
    position_sizing_params=pSizeParamsV1,  # Define as needed
    incl_assets=['NQ1 Index', 'US1 Comdty'],  # Include all involved assets
    name_label='TestMRb2',
    strat_descr='RSI MR basic with 50 SMA trend filter'
)


tRuleMRbasic_os = TradingRule(
    market_data=market_data,
    trading_rule_function=ratioMR_rsi_long2,
    trading_params=strategy_params4,
    position_sizing_params=pSizeParamsV1,  # Define as needed
    incl_assets=['NQ1 Index', 'US1 Comdty'],  # Include all involved assets
    name_label='TestMRb3',
    strat_descr='RSI MR basic with oversold condition'
)

tRuleMRbasic_ostf = TradingRule(
    market_data=market_data,
    trading_rule_function=ratioMR_rsi_long2,
    trading_params=strategy_params5,
    position_sizing_params=pSizeParamsV1,  # Define as needed
    incl_assets=['NQ1 Index', 'US1 Comdty'],  # Include all involved assets
    name_label='TestMRb4',
    strat_descr='RSI MR basic with os and tf'
)



2024-10-16 20:16:21,466 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Initialized TradingRule for function 'ratioMR_rsi_long' with params hash '253043764e'
2024-10-16 20:16:21,467 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Using scalar AssetVol: 10000000 for all assets.
2024-10-16 20:16:21,467 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Initialized TradingRule for function 'ratioMR_rsi_long' with params hash '6e49c6f6d9'
2024-10-16 20:16:21,468 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Using scalar AssetVol: 10000000 for all assets.
2024-10-16 20:16:21,469 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Initialized TradingRule for function 'ratioMR_rsi_long' with params hash '1f0552eae2'
2024-10-16 20:16:21,469 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Using scalar AssetVol: 10000000 for all assets.
2024-10-16 20:16:21,470 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Initialized TradingRule for function 'ratioMR_rsi_long2' with params has

In [14]:
tRules = [trading_rule_MR, tRuleMRbasic, tRuleMRbasic_tf, tRuleMRbasic_os, tRuleMRbasic_ostf]

results = {}
statsdf = {}

for tr in tRules:
    res = tr.backtest_all_assets(save=True)
    resdb = tr.plot_equity(byassets=True, totalsys=True, start_date='01011999')
    stats = tr.calculate_statistics(byassets=True, totalsys=True, start_date='01012007')
    results[tr.strat_descr] = resdb
    statsdf[tr.strat_descr] = stats

2024-10-16 20:16:21,931 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Starting backtest on all assets.
2024-10-16 20:16:21,931 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:21,932 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:22,085 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Backtest data for asset 'NQ1 Index' saved to 'g:\Projects\BackTesting1.0\BackTests\ratioMR_rsi_long_253043764e_5264388cae_TestMR\backtest_results\NQ1 Index_backtest.parquet'
2024-10-16 20:16:22,086 - TradingRule_ratioMR_rsi_long_253043764e - INFO - README.txt created at g:\Projects\BackTesting1.0\BackTests\ratioMR_rsi_long_253043764e_5264388cae_TestMR\README.txt
2024-10-16 20:16:22,086 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Completed backtesting for asset 'NQ1 Index'
2024-10-16 20:16:22,117 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Backtest data for asset 'NQ1 Index' saved to 'g:

2024-10-16 20:16:22,467 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Loading backtest results from saved parquet files.
2024-10-16 20:16:22,488 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Generated cumulative PnL DataFrame by assets.
2024-10-16 20:16:22,490 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:22,657 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:22,659 - TradingRule_ratioMR_rsi_long_253043764e - WARNING - No trades found for asset 'US1 Comdty'.
2024-10-16 20:16:22,799 - TradingRule_ratioMR_rsi_long_253043764e - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:22,934 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Starting backtest on all assets.
2024-10-16 20:16:22,935 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:22,935 - TradingRule_ratioMR_r

2024-10-16 20:16:23,496 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Loading backtest results from saved parquet files.
2024-10-16 20:16:23,512 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Generated cumulative PnL DataFrame by assets.
2024-10-16 20:16:23,514 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:23,673 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:23,674 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - WARNING - No trades found for asset 'US1 Comdty'.
2024-10-16 20:16:23,808 - TradingRule_ratioMR_rsi_long_6e49c6f6d9 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:23,944 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Starting backtest on all assets.
2024-10-16 20:16:23,945 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:23,945 - TradingRule_ratioMR_r

2024-10-16 20:16:24,412 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Loading backtest results from saved parquet files.
2024-10-16 20:16:24,427 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Generated cumulative PnL DataFrame by assets.
2024-10-16 20:16:24,430 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:24,569 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:24,570 - TradingRule_ratioMR_rsi_long_1f0552eae2 - WARNING - No trades found for asset 'US1 Comdty'.
2024-10-16 20:16:24,706 - TradingRule_ratioMR_rsi_long_1f0552eae2 - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:24,842 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Starting backtest on all assets.
2024-10-16 20:16:24,843 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:24,843 - TradingRule_ratioMR

2024-10-16 20:16:25,292 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Loading backtest results from saved parquet files.
2024-10-16 20:16:25,308 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Generated cumulative PnL DataFrame by assets.
2024-10-16 20:16:25,310 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:25,451 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:25,452 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - WARNING - No trades found for asset 'US1 Comdty'.
2024-10-16 20:16:25,599 - TradingRule_ratioMR_rsi_long2_3ce9b6c96a - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:25,735 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Starting backtest on all assets.
2024-10-16 20:16:25,735 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Backtesting asset 'NQ1 Index'
2024-10-16 20:16:25,736 - TradingRule_r

2024-10-16 20:16:26,169 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Loading backtest results from saved parquet files.
2024-10-16 20:16:26,278 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Generated cumulative PnL DataFrame by assets.
2024-10-16 20:16:26,281 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:26,458 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Calculating trade statistics using actual trades.
2024-10-16 20:16:26,459 - TradingRule_ratioMR_rsi_long2_8896291c7a - WARNING - No trades found for asset 'US1 Comdty'.
2024-10-16 20:16:26,621 - TradingRule_ratioMR_rsi_long2_8896291c7a - INFO - Calculating trade statistics using actual trades.


In [15]:
# Extract the desired statistics from each dataframe in statsdf

extracted_stats = {key: statsdf[key].loc['NQ1 Index', ['Sharpe Ratio (ann.)', 'Sortino Ratio (ann.)', 'Hit Rate (%)', 'Profit Factor']] for key in statsdf.keys()}
# Convert the dictionary to a dataframe
extracted_stats_df = pd.DataFrame.from_dict(extracted_stats, orient='index')

extracted_stats_df

,Sharpe Ratio (ann.),Sortino Ratio (ann.),Hit Rate (%),Profit Factor
RSI MR with next day lmt order close - 0.7 atr,-0.122395,-0.161448,48.366013,0.804943
RSI MR basic,1.027742,0.567097,68.682505,1.688279
RSI MR basic with 50 SMA trend filter,0.867863,0.377262,72.101449,1.712846
RSI MR basic with oversold condition,0.441479,0.095433,72.413793,1.863165
RSI MR basic with os and tf,0.500978,0.079926,79.545455,2.571660


In [16]:
nq = market_data.get_ticker_data('NQ1 Index').to_pandas()
nq.set_index('Date', inplace=True)

In [ ]:
import plotly.graph_objects as go

# Extract the OHLC data for 'NQ1 Index'
nq_data = nq[['Open', 'High', 'Low', 'Close']]

# Create the OHLC plot
fig = go.Figure(data=[go.Ohlc(
    x=nq_data.index,
    open=nq_data['Open'],
    high=nq_data['High'],
    low=nq_data['Low'],
    close=nq_data['Close'],
    name='NQ1 Index'
)])

# Update layout for better visualization
fig.update_layout(
    title='NQ1 Index OHLC Chart',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)

# Show the plot
fig.show()